In [4]:
import cv2
import numpy as np
import mediapipe as mp
# import argparse
import os
# from datetime import datetime
from scipy.signal import find_peaks
import time
import math
import csv

In [5]:
mpDraw = mp.solutions.drawing_utils
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence = 0.8)

# Drawing style helpers (optional customizations)
DRAWING_SPEC_LANDMARK = mpDraw.DrawingSpec(color=(255,0,0), thickness=2, circle_radius=2)
DRAWING_SPEC_CONNECTION = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)

In [6]:
# requirements = [
#     "opencv-python",
#     "mediapipe",
#     "numpy"
# ]

# with open("requirements.txt", "w") as f:
#     for r in requirements:
#         f.write(r + "\n")

# print("requirements.txt created!")

In [7]:
def cal_posture(lms):
    """
    Heuristic posture score 1–6:
    Uses shoulder–hip alignment to estimate trunk lean and side lean.
    """
    # Indices
    # ls = mp_pose.PoseLandmark.LEFT_SHOULDER.value
    # rs = mp_pose.PoseLandmark.RIGHT_SHOULDER.value
    # lh = mp_pose.PoseLandmark.LEFT_HIP.value
    # rh = mp_pose.PoseLandmark.RIGHT_HIP.value

    left_shoulder = lms[11]
    right_shoulder = lms[12]
    left_hip = lms[23]
    right_hip = lms[24]
    
    # Estimate trunk angle relative to vertical using mid-shoulder & mid-hip
    mid_shoulder = np.array([(left_shoulder.x + right_shoulder.x) / 2,
                             (left_shoulder.y + right_shoulder.y) / 2])
    mid_hip = np.array([(left_hip.x + right_hip.x) / 2,
                        (left_hip.y + right_hip.y) / 2])

    vec = mid_shoulder - mid_hip
    # angle w.r.t vertical (0, -1)
    vertical = np.array([0, -1])
    cos_angle = np.dot(vec, vertical) / (np.linalg.norm(vec) * np.linalg.norm(vertical) + 1e-6)
    trunk_angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))  # 0 = perfectly vertical

    # Side lean: difference in shoulder heights
    shoulder_y_diff = abs(left_shoulder.y - right_shoulder.y)

    return trunk_angle, shoulder_y_diff


In [8]:
def score_posture(trunk_angle, shoulder_y_diff):
    # Now convert to 1–6 using rough thresholds
    # smaller trunk_angle and small shoulder_y_diff = better
    if trunk_angle <= 10 and shoulder_y_diff <= 0.01:
        return 6  # almost perfect
    elif trunk_angle <= 15 and shoulder_y_diff <= 0.02:
        return 5
    elif trunk_angle <= 20 and shoulder_y_diff <= 0.03:
        return 4
    elif trunk_angle <= 30 and shoulder_y_diff <= 0.05:
        return 3
    elif trunk_angle <= 40:
        return 2
    else:
        return 1

In [9]:
def calc_angle(a, b, c):
    """
    Calculates the angle at point b given three mediapipe landmarks (a, b, c).
    Returns angle in degrees.
    """
    a = np.array([a.x, a.y])
    b = np.array([b.x, b.y])
    c = np.array([c.x, c.y])

    ba = a - b
    bc = c - b

    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
    return angle

In [10]:
def cal_arm_movement(lms):
    """
    Heuristic arm movement score 1–6:
    Uses elbow angles and how much hands cross body midline.
    """
    l_sh = lms[11]
    r_sh = lms[12]
    l_el = lms[13]
    r_el = lms[14]
    l_wr = lms[15]
    r_wr = lms[16]

    # Elbow angles
    left_elbow_angle = calc_angle(l_sh, l_el, l_wr)
    right_elbow_angle = calc_angle(r_sh, r_el, r_wr)

    # Midline x (average of shoulders)
    mid_x = (l_sh.x + r_sh.x) / 2.0

    # Hands crossing midline (distance from own side)
    left_cross = max(0.0, mid_x - l_wr.x)   # if left wrist crosses to right
    right_cross = max(0.0, r_wr.x - mid_x)  # if right wrist crosses to left

    avg_elbow_angle = (left_elbow_angle + right_elbow_angle) / 2.0
    cross_amount = left_cross + right_cross

    return avg_elbow_angle, cross_amount

In [11]:
def score_arm_movement(avg_elbow_angle, cross_amount):
    # Ideal elbow angle ~90°, allow ±20°
    # Less crossing = better
    if 70 <= avg_elbow_angle <= 110 and cross_amount < 0.01:
        return 6
    elif 60 <= avg_elbow_angle <= 120 and cross_amount < 0.02:
        return 5
    elif 50 <= avg_elbow_angle <= 130 and cross_amount < 0.03:
        return 4
    elif 40 <= avg_elbow_angle <= 140 and cross_amount < 0.05:
        return 3
    elif 30 <= avg_elbow_angle <= 150:
        return 2
    else:
        return 1

In [12]:
def cal_leg_action(lms):
    """
    Heuristic leg action score 1–6:
    Uses hip-knee-ankle angles and knee height relative to hip.
    """
    lh = lms[23]
    rh = lms[24]
    lk = lms[25]
    rk = lms[26]
    la = lms[27]
    ra = lms[28]

    # Knee angles (stance/drive quality)
    left_knee_angle = calc_angle(lh, lk, la)
    right_knee_angle = calc_angle(rh, rk, ra)

    # Knee height relative to hip (inverted because y increases downward)
    left_knee_lift = lh.y - lk.y
    right_knee_lift = rh.y - rk.y
    knee_lift = max(left_knee_lift, right_knee_lift)

    avg_knee_angle = (left_knee_angle + right_knee_angle) / 2.0

    return knee_lift, avg_knee_angle

In [13]:
def score_leg_action(knee_lift, avg_knee_angle):
    # Heuristic thresholds:
    # - Knee flexion/extension around 160–180° at push-off,
    # - Knee lift (hip_y - knee_y) being reasonably positive
    if knee_lift >= 0.10 and 150 <= avg_knee_angle <= 180:
        return 6
    elif knee_lift >= 0.08 and 140 <= avg_knee_angle <= 180:
        return 5
    elif knee_lift >= 0.06 and 130 <= avg_knee_angle <= 180:
        return 4
    elif knee_lift >= 0.04 and 120 <= avg_knee_angle <= 180:
        return 3
    elif knee_lift >= 0.02:
        return 2
    else:
        return 1

In [14]:
def compute_sprint_time_from_frames(start_frame_idx: int, finish_frame_idx: int, fps: float) -> float:
    """
    Compute sprint time using start and finish frame indices and FPS.
    time = (finish - start) / fps
    """
    if fps <= 0:
        raise ValueError("FPS must be > 0")
    return (finish_frame_idx - start_frame_idx) / fps


In [15]:
def score_sprint_time(time_sec: float) -> int:
    """
    Map 20m sprint time (in seconds) to Likert score 1–5 based on your matrix.
    """
    if time_sec <= 4.1:
        return 5
    elif time_sec <= 4.5:
        return 4
    elif time_sec <= 4.9:
        return 3
    elif time_sec <= 5.3:
        return 2
    else:
        return 1


In [16]:
def predict_category(_score, sprint_score):
    
    # Safety checks
    if _score < 0: _score = 0
    if _score > 20: _score = 20
    if sprint_score < 0: sprint_score = 0
    if sprint_score > 5: sprint_score = 5

    # -------------------------------
    # Top Tier Category (Excellent)
    # -------------------------------
    if _score >= 17:
        if sprint_score >= 4:
            return "Excellent"
        else:
            return "Above Average"

    # -------------------------------
    # High-Mid Tier (Above Average / Average)
    # -------------------------------
    elif 14 <= _score <= 17:
        if sprint_score >= 3:
            return "Above Average"
        elif sprint_score == 2:
            return "Average"
        else:
            return "Below Average"

    # -------------------------------
    # Mid Tier (Average / Below Average)
    # -------------------------------
    elif 10 <= _score <= 13:
        if sprint_score >= 3:
            return "Average"
        elif sprint_score == 2:
            return "Below Average"
        else:
            return "Poor"

    # -------------------------------
    # Lower Tier (Below Average)
    # -------------------------------
    elif 6 <= _score <= 9:
        if sprint_score >= 2:
            return "Below Average"
        else:
            return "Poor"

    # -------------------------------
    # Lowest Tier (Poor)
    # -------------------------------
    else:  # _score 0–5
        return "Poor"

In [17]:
def add_data(row):
    # row must be a list: ["value1", "value2", ...]
    with open("meter_run_results.csv", "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(row)

In [18]:
open("meter_run_results.csv", "w").close()
data = ["ID", "Name", "Video_path", "Sprint_time", "Trunk_angle_avg", "Shoulder_y_diff_avg", "Avg_elbow_angle_avg", "Cross_amount_avg", "Knee_lift_avg", "Avg_knee_angle_avg", "Score_18", "Category"]
add_data(data)

In [59]:
def meter_run(ID="DCXXXX", name="Life", path=0, data_print='Y'):
    
    # cap = cv2.VideoCapture(0)  # 0 = default camera
    # path = "img-5610-0q6jn2tf_fZX3eQZg.mov"
    cap = cv2.VideoCapture(path)


    # Define video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # remove in live case
    fps = cap.get(cv2.CAP_PROP_FPS)                            # frames per second
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))       # total frames
    # video_time = frame_count / fps

    # print("FPS:", fps)
    # print("Total Frames:", frame_count)
    # print("Video Duration (seconds):", video_time_seconds)


    # save the output video
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    # out = cv2.VideoWriter(video_filename, fourcc, fps, (frame_width, frame_height))

    left_line_x = 0.05
    right_line_x = 0.87

    # posture const
    trunk_angle, shoulder_y_diff = 0, 0
    p_score, posture_final = 0, 0

    # arms const
    avg_elbow_angle, cross_amount = 0, 0
    a_score, arms_final = 0, 0

    # leg const
    knee_lift, avg_knee_angle = 0, 0
    l_score, legs_final = 0, 0

    # all pose and pose-score append here     # will store (posture, arms, legs)
    pose_all = []
    pose_scores = [] 

    # Sprint time + score
    sprint_time = []
    sprint_score = 0

    # change acc to your threshold
    start_frame_idx = 0      # example 
    finish_frame_idx = fps   # example

    # total score
    _score = 0

    frame_idx = 0
    start_time = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        # Convert BGR → RGB for MediaPipe
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Run pose detection
        results = pose.process(rgb)

        if results.pose_landmarks:
            # Enumerate all landmarks
            # for id, lm in enumerate(results.pose_landmarks.landmark):
            #     # Optional: Draw skeleton
            #     mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)
                
            #     # Convert normalized landmark to pixel coordinates
            #     h, w, c = frame.shape
            #     cx, cy = int(lm.x * w), int(lm.y * h)
            #     # IDs to highlight: 23, 24, 25, 26
            #     if id in [23, 24, 25, 26]:
            #         # Draw circle on frame
            #         cv2.circle(frame, (cx, cy), 8, (0, 255, 0), -1)
            
        
            mpDraw.draw_landmarks(frame, results.pose_landmarks, mpPose.POSE_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)
            lms = results.pose_landmarks.landmark

            h, w, c = frame.shape
            cx1 = int(left_line_x * w)
            cx2 = int(right_line_x * w)
            cy1 = int(0.48 * h)
            cy2 = int(0.48 * h)
            cv2.circle(frame, (cx1, cy1), 8, (0, 255, 0), -1)
            cv2.circle(frame, (cx2, cy2), 8, (0, 255, 0), -1)
            cx = int((cx1+cx2)/2)
            cy = int((cy1+cy2)/2)
            cv2.circle(frame, (cx, cy), 8, (0, 0, 255), -1)
   
            # posture
            trunk_angle, shoulder_y_diff = cal_posture(lms)
            p_score = score_posture(trunk_angle, shoulder_y_diff)

            # arm
            avg_elbow_angle, cross_amount = cal_arm_movement(lms)
            a_score = score_arm_movement(avg_elbow_angle, cross_amount)

            # leg
            knee_lift, avg_knee_angle = cal_leg_action(lms)
            l_score = score_leg_action(knee_lift, avg_knee_angle)

            # append all values
            pose_all.append((trunk_angle, shoulder_y_diff, avg_elbow_angle, cross_amount, knee_lift, avg_knee_angle))
            pose_scores.append((p_score, a_score, l_score))

            # Optional: draw text on frame
            cv2.putText(frame, f"P: trunk_angle: {trunk_angle:.1f}, shoulder_y_diff: {shoulder_y_diff:.1f}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"A: avg_elbow_angle: {avg_elbow_angle:.1f}, cross_amount: {cross_amount:.1f}", (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"L: knee_lift: {knee_lift:.1f}, avg_knee_angle: {avg_knee_angle:.1f}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)
            

            # out.write(frame)
            win_name = "Resizable Window"
            cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
            cv2.imshow(win_name, frame)

            # cv2.imshow("Press 'q' to stop early", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    # out.release()
    cv2.destroyAllWindows()

    # Aggregate all pose
    if pose_all:
        trunk_angle_avg = round(np.mean([a for a, _, _, _, _, _ in pose_all]), 2)
        shoulder_y_diff_avg = round(np.mean([b for _, b, _, _, _, _ in pose_all]), 2)
        avg_elbow_angle_avg = round(np.mean([c for _, _, c, _, _, _ in pose_all]), 2)
        cross_amount_avg = round(np.mean([d for _, _, _, d, _, _ in pose_all]), 2)
        knee_lift_avg = round(np.mean([e for _, _, _, _, e, _ in pose_all]), 2)
        avg_knee_angle_avg = round(np.mean([f for f, _, _, _, _, e in pose_all]), 2)


    # Aggregate technique scores
    if pose_scores:
        posture_avg_score = np.mean([p for p, _, _ in pose_scores])
        arms_avg_score = np.mean([a for _, a, _ in pose_scores])
        legs_avg_score = np.mean([l for _, _, l in pose_scores])

        # Total score / 18
        _score = round((posture_avg_score + arms_avg_score + legs_avg_score), 2) 

    # Sprint time + score
    sprint_time = compute_sprint_time_from_frames(start_frame_idx, finish_frame_idx, fps)
    sprint_score = score_sprint_time(sprint_time)

    category = predict_category(_score, sprint_score)

    # show data
    if (data_print =='Y' or data_print == 'y'):
        print(f"Sprint time: {sprint_time:.2f} s → Time score (1–5): {sprint_score}")
        print(f"Technique – posture:--> trunk_angle_avg: {trunk_angle_avg}, shoulder_y_diff_avg: {shoulder_y_diff_avg}")
        print(f"Technique – arms:--> avg_elbow_angle_avg: {avg_elbow_angle_avg}, cross_amount_avg: {cross_amount_avg}")
        print(f"Technique – legs:--> knee_lift_avg: {knee_lift_avg}, avg_knee_angle_avg: {avg_knee_angle_avg}")
        print(f"Technique total points: {_score}/18 and Category:--> {category}")
        

    # adding data
    data = [ID, name, path, sprint_time, trunk_angle_avg, shoulder_y_diff_avg, avg_elbow_angle_avg, cross_amount_avg, knee_lift_avg, avg_knee_angle_avg, _score, category]
    add_data(data)

    # print(_score, "--", category)
    return _score, category


In [60]:
# ID = input("(DCXXXXX)Enter unique ID:")
# name = input("Name of Candidate:")
# path = input("Path of your Video:")
# # "demo-shuttle-run_HwmEdnDL.mp4"
# data_print = input("Wants to print data Y/N:")
# shuttle_score, category = shuttle_run(ID, name, path, data_print)
# print("printing result here-------------->>")
# print(f"Your High Knee Jump Score is--> {shuttle_score:.2f}/18 and Category--> {category}")

In [65]:
ID = "DC0001"
name = "Nayan"
# path = "20m_running/20m Shuttle Run Test_720p (online-video-cutter.com).mp4"
# path = "Test_Run.mp4"
path = "IMG_5805 (online-video-cutter.com).mp4"
data_print = "Y"
# data_print = input("Wants to print data Y/N:")
meter_score, category = meter_run(ID, name, path, data_print)

Sprint time: 1.00 s → Time score (1–5): 5
Technique – posture:--> trunk_angle_avg: 9.98, shoulder_y_diff_avg: 0.01
Technique – arms:--> avg_elbow_angle_avg: 139.26, cross_amount_avg: 0.07
Technique – legs:--> knee_lift_avg: -0.07, avg_knee_angle_avg: 9.98
Technique total points: 8.36/18 and Category:--> Below Average
